# Atasözleri Scrape

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re # Sayfa navigasyonu için

# --- 1. Sabit Tanımlamalar ---
ANA_URL = "https://www.turkcedersi.net/atasozleri-ve-atasozlerinin-anlamlari/"
HARFLER = ['a', 'b', 'c', 'ç', 'd', 'e', 'f', 'g', 'ğ', 'h', 'ı', 'i', 'j',
           'k', 'l', 'm', 'n', 'o', 'ö', 'p', 'r', 's', 'ş', 't', 'u', 'ü', 'v', 'y', 'z']
CSV_DOSYA_ADI = 'atasozleri_ve_deyimler_sayfali.csv'

# --- 2. CSV Dosyasını Oluşturma ve Başlıkları Yazma ---
with open(CSV_DOSYA_ADI, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter=';')
    writer.writerow(['Baslik', 'Aciklama'])

# --- 3. Harf ve Sayfa Döngüsü ile Scraping İşlemi ---
for harf in HARFLER:
    page_num = 1
    has_next_page = True

    while has_next_page:
        VERI = []

        # URL'yi oluşturma: page/1/?_harf=a, page/2/?_harf=a, ...
        # İlk sayfa için /page/1/ kısmını atlayarak temiz bir URL kullanıyoruz.
        if page_num == 1:
            FULL_URL = f"{ANA_URL}?_harf={harf}"
        else:
            FULL_URL = f"{ANA_URL}page/{page_num}/?_harf={harf}"

        print(f"-> {harf.upper()} harfi için {page_num}. sayfa çekiliyor...")

        try:
            # Sayfayı çek
            response = requests.get(FULL_URL)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # --- 3a. Veri Çekme (Önceki Adımlardaki Yapı) ---

            # Tüm atasözü/deyim kayıtlarını toplama (h2 ve p etiketleri ile)
            atasozleri = soup.find_all('h2')
            aciklamalar = soup.find_all('p')

            for i in range(min(len(atasozleri), len(aciklamalar))):
                baslik = atasozleri[i].text.strip()
                aciklama = aciklamalar[i].text.strip()

                # Basit bir filtreleme (Sayfa başlıkları vb. atlamak için)
                if baslik and len(baslik) > 5 and not baslik.startswith("SİVAS"):
                    VERI.append([baslik, aciklama])

            # Toplanan veriyi dosyaya ekle
            with open(CSV_DOSYA_ADI, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f, delimiter=';')
                writer.writerows(VERI)

            # --- 3b. Sonraki Sayfa Kontrolü (Sayfalama Mantığı) ---

            # Sayfalama butonu XPath'i: //div[@class='pagination']/a
            # Bu, pagination sınıfındaki tüm linkleri arayacak.
            # Sayfalama navigasyonu genellikle 'pagination' veya 'page-nav' gibi bir sınıfta bulunur.

            # Sayfalama div'ini bulmaya çalışalım (Sayfalama linklerini içerir)
            pagination_div = soup.find('div', class_='pagination')

            # Eğer pagination div'i varsa, içinde 'Sonraki' veya '>' gibi bir link ararız.
            if pagination_div:
                # 'Sonraki' kelimesi içeren bir link var mı kontrol et.
                next_link = pagination_div.find('a', string=re.compile(r'Sonraki|»|›'))

                if next_link:
                    # 'Sonraki' linki bulunduysa, bir sonraki sayfa var demektir.
                    page_num += 1
                else:
                    # 'Sonraki' linki yoksa, bu harfin son sayfasıdır.
                    has_next_page = False
            else:
                # Pagination div'i yoksa, bu sayfa tek sayfadır.
                has_next_page = False

        except requests.exceptions.HTTPError as e:
            # 404 Not Found hatası alınırsa (sayfa yoksa)
            print(f"Hata: {FULL_URL} adresi bulunamadı. ({e}). Sayfalama bitiriliyor.")
            has_next_page = False

        except requests.exceptions.RequestException as e:
            print(f"Bağlantı Hatası ({harf.upper()} - Sayfa {page_num}): {e}. İşlem durduruluyor.")
            has_next_page = False

        # Sunucuyu yormamak için kısa bir bekleme (Çok önemlidir!)
        time.sleep(1)

    print(f"--- {harf.upper()} harfi için tüm sayfalar çekildi. ---")

print("\n--- TÜM VERİ ÇEKİM İŞLEMİ TAMAMLANDI! ---")

## Deyimler Scrape

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re

# --- 1. Sabit Tanımlamalar ---
# Deyimler için ANA URL'yi güncelliyoruz.
ANA_URL = "https://www.turkcedersi.net/deyimler-ve-deyimlerin-anlamlari/"

# Sayfalama yapısı: Deyimler için de 'page/N/' yapısı ve '?_harf=x' parametresi kullanılacak.
HARFLER = ['a', 'b', 'c', 'ç', 'd', 'e', 'f', 'g', 'ğ', 'h', 'ı', 'i', 'j',
           'k', 'l', 'm', 'n', 'o', 'ö', 'p', 'r', 's', 'ş', 't', 'u', 'ü', 'v', 'y', 'z']
CSV_DOSYA_ADI = 'deyimler_ve_anlamlari_sayfali.csv'

# --- 2. CSV Dosyasını Oluşturma ve Başlıkları Yazma ---
with open(CSV_DOSYA_ADI, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter=';')
    writer.writerow(['Deyim', 'Anlami'])

# --- 3. Harf ve Sayfa Döngüsü ile Scraping İşlemi ---
for harf in HARFLER:
    page_num = 1
    has_next_page = True

    while has_next_page:
        VERI = []

        # URL'yi oluşturma: Deyimler için de 'page/N/?_harf=x' yapısını kullanıyoruz.
        if page_num == 1:
            FULL_URL = f"{ANA_URL}?_harf={harf}"
        else:
            # Deyimler sayfasının URL yapısı atasözleri sayfasının yapısıyla aynı olmalıdır.
            # Örneğin: https://www.turkcedersi.net/deyimler-ve-deyimlerin-anlamlari/page/2/?_harf=a
            FULL_URL = f"{ANA_URL}page/{page_num}/?_harf={harf}"

        print(f"-> {harf.upper()} harfi için {page_num}. sayfa çekiliyor...")

        try:
            # Sayfayı çek
            response = requests.get(FULL_URL)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # --- 3a. Veri Çekme (Atasözleri ile Aynı Etiket Yapısı Kullanılıyor) ---

            # Tüm deyim kayıtlarını toplama (h2 ve p etiketleri ile)
            deyimbasliklari = soup.find_all('h2')
            anlamlar = soup.find_all('p')

            for i in range(min(len(deyimbasliklari), len(anlamlar))):
                baslik = deyimbasliklari[i].text.strip()
                anlam = anlamlar[i].text.strip()

                # Basit bir filtreleme
                if baslik and len(baslik) > 5 and not baslik.startswith("SİVAS"):
                    VERI.append([baslik, anlam])

            # Toplanan veriyi dosyaya ekle
            with open(CSV_DOSYA_ADI, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f, delimiter=';')
                writer.writerows(VERI)

            # --- 3b. Sonraki Sayfa Kontrolü ---

            # Sayfalama div'ini bulmaya çalışalım (Sayfalama linklerini içerir)
            pagination_div = soup.find('div', class_='pagination')

            if pagination_div:
                # 'Sonraki' kelimesi içeren bir link var mı kontrol et.
                next_link = pagination_div.find('a', string=re.compile(r'Sonraki|»|›'))

                if next_link:
                    # 'Sonraki' linki bulunduysa, bir sonraki sayfa var demektir.
                    page_num += 1
                else:
                    # 'Sonraki' linki yoksa, bu harfin son sayfasıdır.
                    has_next_page = False
            else:
                # Pagination div'i yoksa, bu sayfa tek sayfadır.
                has_next_page = False

        except requests.exceptions.HTTPError as e:
            # 404 Not Found hatası alınırsa (sayfa yoksa)
            print(f"Hata: {FULL_URL} adresi bulunamadı. ({e}). Sayfalama bitiriliyor.")
            has_next_page = False

        except requests.exceptions.RequestException as e:
            print(f"Bağlantı Hatası ({harf.upper()} - Sayfa {page_num}): {e}. İşlem durduruluyor.")
            has_next_page = False

        # Sunucuyu yormamak için kısa bir bekleme (Çok önemlidir!)
        time.sleep(1)

    print(f"--- {harf.upper()} harfi için tüm sayfalar çekildi. ---")

print("\n--- TÜM DEYİMLERİN ÇEKİM İŞLEMİ TAMAMLANDI! ---")

# TANIMLAR VE ÖRNEKLER AYRI SÜTUNLARA ALINIYOR. ÖYLE ÇEKİLİYOR.

## Atasözü

In [2]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re

# --- 1. Sabit Tanımlamalar ---
ANA_URL = "https://www.turkcedersi.net/atasozleri-ve-atasozlerinin-anlamlari/" # VEYA atasozleri-ve-anlamlari URL'si

HARFLER = ['a', 'b', 'c', 'ç', 'd', 'e', 'f', 'g', 'ğ', 'h', 'ı', 'i', 'j',
           'k', 'l', 'm', 'n', 'o', 'ö', 'p', 'r', 's', 'ş', 't', 'u', 'ü', 'v', 'y', 'z']
CSV_DOSYA_ADI = 'atasozleri_coklu.csv'

# --- 2. CSV Dosyasını Oluşturma ve BAŞLIKLARI YAZMA ---
with open(CSV_DOSYA_ADI, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter=';')
    # Yeni format: Başlık, Tanım Numarası, Tanım, Örnek
    writer.writerow(['Deyim', 'Tanım_No', 'Tanım', 'Örnek'])

# --- 3. Yardımcı Fonksiyon: Çoklu Tanımı Ayrıştırma ---
def parse_multi_definition(deyimbasligi, anlam_metni):
    kayitlar = []

    # Adım A: Metni (1), (2), (3) gibi numaralara göre böl.
    # Bu, tüm metni, tanım numaralarının başına bir işaret koyarak bölecek.
    # Örn: ['Metin', '1) belli olmak... -A. Ümit. 2) rıhtıma aborda...']
    parcalar = re.split(r'(\d+\-\s?|(?=\d+\)))', anlam_metni.strip())

    # İlk parça genellikle boş veya gereksiz metin olabilir, atılır.
    parcalar = [p.strip() for p in parcalar if p.strip()]

    # Adım B: Ayrılmış parçaları Tanım ve Örnek olarak işle
    tanim_no = 1

    # Numaralandırma varsa (1, 2, 3...) bu döngü çalışır
    if any(re.match(r'^\d+\)', p) for p in parcalar):
        # Eğer parçalar 1), 2) gibi başlıyorsa, bunlar ana tanım bloklarıdır.

        # 1), 2) gibi numaralandırmayı kullanarak tanım bloklarını grupla
        tanim_bloklari = []
        gecici_blok = ""
        for p in parcalar:
            if re.match(r'^\d+\)', p):
                if gecici_blok:
                    tanim_bloklari.append(gecici_blok.strip())
                gecici_blok = p
            else:
                gecici_blok += p
        if gecici_blok:
            tanim_bloklari.append(gecici_blok.strip())

        for blok in tanim_bloklari:
            # Örnek cümleyi ayırmak için tırnak içindeki kısmı bul
            # Örnek: '...metin' –Yazar. gibi yapıları bulmaya çalış.

            # Tanım numarasını metin başından temizle
            blok_temiz = re.sub(r'^\d+\)\s?', '', blok, 1).strip()

            # Örnek tırnak içinde ve sonunda yazar adı ile bitiyorsa
            ornek_match = re.search(r'(\‘(.*?)\’\s*–[A-ZÖÇŞIĞÜ.][\w\s.-]+)', blok_temiz)

            ornek_cumle = ""
            tanim_metni = blok_temiz

            if ornek_match:
                # Örnek cümleyi ve yazar adını al
                ornek_cumle = ornek_match.group(1).strip()
                # Tanım metninden örnek cümleyi çıkar
                tanim_metni = blok_temiz.replace(ornek_match.group(0), '').strip()

            # İki nokta üst üste varsa onu da temizle (genellikle tanım sonunda)
            tanim_metni = tanim_metni.rstrip(':').strip()

            # Yeni kaydı ekle
            kayitlar.append([deyimbasligi, tanim_no, tanim_metni, ornek_cumle])
            tanim_no += 1

    # Numaralandırma yoksa (Tek tanım varsayımı)
    else:
        # Tek tanım olduğu varsayılır, basit ayırma yapılır
        ornek_match = re.search(r'(\‘(.*?)\’\s*–[A-ZÖÇŞIĞÜ.][\w\s.-]+)', anlam_metni)

        ornek_cumle = ""
        tanim_metni = anlam_metni

        if ornek_match:
            ornek_cumle = ornek_match.group(1).strip()
            tanim_metni = anlam_metni.replace(ornek_match.group(0), '').strip()

        tanim_metni = tanim_metni.rstrip(':').strip()

        kayitlar.append([deyimbasligi, 1, tanim_metni, ornek_cumle])

    return kayitlar

# --- 4. Harf ve Sayfa Döngüsü ile Scraping İşlemi ---
for harf in HARFLER:
    page_num = 1
    has_next_page = True

    while has_next_page:
        VERI = []

        if page_num == 1:
            FULL_URL = f"{ANA_URL}?_harf={harf}"
        else:
            FULL_URL = f"{ANA_URL}page/{page_num}/?_harf={harf}"

        print(f"-> {harf.upper()} harfi için {page_num}. sayfa çekiliyor...")

        try:
            response = requests.get(FULL_URL)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            deyimbasliklari = soup.find_all('h2')
            anlamlar = soup.find_all('p')

            for i in range(min(len(deyimbasliklari), len(anlamlar))):
                baslik = deyimbasliklari[i].text.strip()
                anlam = anlamlar[i].text.strip()

                # *** PARÇALAMA FONKSİYONUNU KULLAN ***
                tanim_kayitlari = parse_multi_definition(baslik, anlam)
                VERI.extend(tanim_kayitlari)

            # Toplanan veriyi dosyaya ekle
            with open(CSV_DOSYA_ADI, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f, delimiter=';')
                writer.writerows(VERI)

            # Sonraki Sayfa Kontrolü (Aynı Mantık)
            pagination_div = soup.find('div', class_='pagination')

            if pagination_div:
                next_link = pagination_div.find('a', string=re.compile(r'Sonraki|»|›'))
                if next_link:
                    page_num += 1
                else:
                    has_next_page = False
            else:
                has_next_page = False

        except requests.exceptions.HTTPError as e:
            print(f"Hata: {FULL_URL} adresi bulunamadı. ({e}). Sayfalama bitiriliyor.")
            has_next_page = False

        except requests.exceptions.RequestException as e:
            print(f"Bağlantı Hatası ({harf.upper()} - Sayfa {page_num}): {e}. İşlem durduruluyor.")
            has_next_page = False

        time.sleep(1)

    print(f"--- {harf.upper()} harfi için tüm sayfalar çekildi. ---")

print("\n--- ÇOKLU TANIM ÇEKİM İŞLEMİ TAMAMLANDI! ---")

-> A harfi için 1. sayfa çekiliyor...
-> A harfi için 2. sayfa çekiliyor...
-> A harfi için 3. sayfa çekiliyor...
-> A harfi için 4. sayfa çekiliyor...
-> A harfi için 5. sayfa çekiliyor...
--- A harfi için tüm sayfalar çekildi. ---
-> B harfi için 1. sayfa çekiliyor...
-> B harfi için 2. sayfa çekiliyor...
-> B harfi için 3. sayfa çekiliyor...
--- B harfi için tüm sayfalar çekildi. ---
-> C harfi için 1. sayfa çekiliyor...
--- C harfi için tüm sayfalar çekildi. ---
-> Ç harfi için 1. sayfa çekiliyor...
--- Ç harfi için tüm sayfalar çekildi. ---
-> D harfi için 1. sayfa çekiliyor...
-> D harfi için 2. sayfa çekiliyor...
--- D harfi için tüm sayfalar çekildi. ---
-> E harfi için 1. sayfa çekiliyor...
-> E harfi için 2. sayfa çekiliyor...
--- E harfi için tüm sayfalar çekildi. ---
-> F harfi için 1. sayfa çekiliyor...
--- F harfi için tüm sayfalar çekildi. ---
-> G harfi için 1. sayfa çekiliyor...
-> G harfi için 2. sayfa çekiliyor...
--- G harfi için tüm sayfalar çekildi. ---
-> Ğ harfi

## Deyim

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re

# --- 1. Sabit Tanımlamalar ---
ANA_URL = "https://www.turkcedersi.net/deyimler-ve-deyimlerin-anlamlari/" # VEYA atasozleri-ve-anlamlari URL'si

HARFLER = ['a', 'b', 'c', 'ç', 'd', 'e', 'f', 'g', 'ğ', 'h', 'ı', 'i', 'j',
           'k', 'l', 'm', 'n', 'o', 'ö', 'p', 'r', 's', 'ş', 't', 'u', 'ü', 'v', 'y', 'z']
CSV_DOSYA_ADI = 'deyimler_coklu.csv'

# --- 2. CSV Dosyasını Oluşturma ve BAŞLIKLARI YAZMA ---
with open(CSV_DOSYA_ADI, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter=';')
    # Yeni format: Başlık, Tanım Numarası, Tanım, Örnek
    writer.writerow(['Deyim', 'Tanım_No', 'Tanım', 'Örnek'])

# --- 3. Yardımcı Fonksiyon: Çoklu Tanımı Ayrıştırma ---
def parse_multi_definition(deyimbasligi, anlam_metni):
    kayitlar = []

    # Adım A: Metni (1), (2), (3) gibi numaralara göre böl.
    # Bu, tüm metni, tanım numaralarının başına bir işaret koyarak bölecek.
    # Örn: ['Metin', '1) belli olmak... -A. Ümit. 2) rıhtıma aborda...']
    parcalar = re.split(r'(\d+\-\s?|(?=\d+\)))', anlam_metni.strip())

    # İlk parça genellikle boş veya gereksiz metin olabilir, atılır.
    parcalar = [p.strip() for p in parcalar if p.strip()]

    # Adım B: Ayrılmış parçaları Tanım ve Örnek olarak işle
    tanim_no = 1

    # Numaralandırma varsa (1, 2, 3...) bu döngü çalışır
    if any(re.match(r'^\d+\)', p) for p in parcalar):
        # Eğer parçalar 1), 2) gibi başlıyorsa, bunlar ana tanım bloklarıdır.

        # 1), 2) gibi numaralandırmayı kullanarak tanım bloklarını grupla
        tanim_bloklari = []
        gecici_blok = ""
        for p in parcalar:
            if re.match(r'^\d+\)', p):
                if gecici_blok:
                    tanim_bloklari.append(gecici_blok.strip())
                gecici_blok = p
            else:
                gecici_blok += p
        if gecici_blok:
            tanim_bloklari.append(gecici_blok.strip())

        for blok in tanim_bloklari:
            # Örnek cümleyi ayırmak için tırnak içindeki kısmı bul
            # Örnek: '...metin' –Yazar. gibi yapıları bulmaya çalış.

            # Tanım numarasını metin başından temizle
            blok_temiz = re.sub(r'^\d+\)\s?', '', blok, 1).strip()

            # Örnek tırnak içinde ve sonunda yazar adı ile bitiyorsa
            ornek_match = re.search(r'(\‘(.*?)\’\s*–[A-ZÖÇŞIĞÜ.][\w\s.-]+)', blok_temiz)

            ornek_cumle = ""
            tanim_metni = blok_temiz

            if ornek_match:
                # Örnek cümleyi ve yazar adını al
                ornek_cumle = ornek_match.group(1).strip()
                # Tanım metninden örnek cümleyi çıkar
                tanim_metni = blok_temiz.replace(ornek_match.group(0), '').strip()

            # İki nokta üst üste varsa onu da temizle (genellikle tanım sonunda)
            tanim_metni = tanim_metni.rstrip(':').strip()

            # Yeni kaydı ekle
            kayitlar.append([deyimbasligi, tanim_no, tanim_metni, ornek_cumle])
            tanim_no += 1

    # Numaralandırma yoksa (Tek tanım varsayımı)
    else:
        # Tek tanım olduğu varsayılır, basit ayırma yapılır
        ornek_match = re.search(r'(\‘(.*?)\’\s*–[A-ZÖÇŞIĞÜ.][\w\s.-]+)', anlam_metni)

        ornek_cumle = ""
        tanim_metni = anlam_metni

        if ornek_match:
            ornek_cumle = ornek_match.group(1).strip()
            tanim_metni = anlam_metni.replace(ornek_match.group(0), '').strip()

        tanim_metni = tanim_metni.rstrip(':').strip()

        kayitlar.append([deyimbasligi, 1, tanim_metni, ornek_cumle])

    return kayitlar

# --- 4. Harf ve Sayfa Döngüsü ile Scraping İşlemi ---
for harf in HARFLER:
    page_num = 1
    has_next_page = True

    while has_next_page:
        VERI = []

        if page_num == 1:
            FULL_URL = f"{ANA_URL}?_harf={harf}"
        else:
            FULL_URL = f"{ANA_URL}page/{page_num}/?_harf={harf}"

        print(f"-> {harf.upper()} harfi için {page_num}. sayfa çekiliyor...")

        try:
            response = requests.get(FULL_URL)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            deyimbasliklari = soup.find_all('h2')
            anlamlar = soup.find_all('p')

            for i in range(min(len(deyimbasliklari), len(anlamlar))):
                baslik = deyimbasliklari[i].text.strip()
                anlam = anlamlar[i].text.strip()

                # *** PARÇALAMA FONKSİYONUNU KULLAN ***
                tanim_kayitlari = parse_multi_definition(baslik, anlam)
                VERI.extend(tanim_kayitlari)

            # Toplanan veriyi dosyaya ekle
            with open(CSV_DOSYA_ADI, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f, delimiter=';')
                writer.writerows(VERI)

            # Sonraki Sayfa Kontrolü (Aynı Mantık)
            pagination_div = soup.find('div', class_='pagination')

            if pagination_div:
                next_link = pagination_div.find('a', string=re.compile(r'Sonraki|»|›'))
                if next_link:
                    page_num += 1
                else:
                    has_next_page = False
            else:
                has_next_page = False

        except requests.exceptions.HTTPError as e:
            print(f"Hata: {FULL_URL} adresi bulunamadı. ({e}). Sayfalama bitiriliyor.")
            has_next_page = False

        except requests.exceptions.RequestException as e:
            print(f"Bağlantı Hatası ({harf.upper()} - Sayfa {page_num}): {e}. İşlem durduruluyor.")
            has_next_page = False

        time.sleep(1)

    print(f"--- {harf.upper()} harfi için tüm sayfalar çekildi. ---")

print("\n--- ÇOKLU TANIM ÇEKİM İŞLEMİ TAMAMLANDI! ---")

-> A harfi için 1. sayfa çekiliyor...
-> A harfi için 2. sayfa çekiliyor...
-> A harfi için 3. sayfa çekiliyor...
-> A harfi için 4. sayfa çekiliyor...
-> A harfi için 5. sayfa çekiliyor...
-> A harfi için 6. sayfa çekiliyor...
-> A harfi için 7. sayfa çekiliyor...
-> A harfi için 8. sayfa çekiliyor...
-> A harfi için 9. sayfa çekiliyor...
-> A harfi için 10. sayfa çekiliyor...
-> A harfi için 11. sayfa çekiliyor...
--- A harfi için tüm sayfalar çekildi. ---
-> B harfi için 1. sayfa çekiliyor...
-> B harfi için 2. sayfa çekiliyor...
-> B harfi için 3. sayfa çekiliyor...
-> B harfi için 4. sayfa çekiliyor...
-> B harfi için 5. sayfa çekiliyor...
-> B harfi için 6. sayfa çekiliyor...
-> B harfi için 7. sayfa çekiliyor...
-> B harfi için 8. sayfa çekiliyor...
-> B harfi için 9. sayfa çekiliyor...
--- B harfi için tüm sayfalar çekildi. ---
-> C harfi için 1. sayfa çekiliyor...
-> C harfi için 2. sayfa çekiliyor...
-> C harfi için 3. sayfa çekiliyor...
--- C harfi için tüm sayfalar çekildi.